# Trying Again

In [ ]:
# packages
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, classification_report

In [ ]:
# step 1: load data
df = pd.read_csv('../data/raw/banksim.csv')

In [ ]:
# step 2: basic cleaning
# drop useless columns
df = df.drop(columns=["zipcodeori", "zipmerchant"])

# remove quotes
for col in ['customer', 'age', 'gender', 'merchant', 'category']:
    df[col] = df[col].astype(str).str.replace("'", "")

# handle uncommon values
df["gender"] = df["gender"].replace(['U', 'E'], np.nan)
df["age"] = df["age"].replace('U', np.nan)

# convert age to numeric
df["age"] = pd.to_numeric(df["age"], errors='coerce')

# fill missing values
df['gender'] = df['gender'].fillna('Unknown')
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)

In [ ]:
# step 3: separate features and target
X = df.drop(columns=["fraud"])
y = df["fraud"] # test

In [ ]:
# step 4: define feature types
numerical_features = ["amount", "age","step"] # to be scaled
categorical_features = ["gender", "category"] # to be one-hot encoded
id_features = ["customer", "merchant"]        # to be encoded as nodes

In [ ]:
# step 5: encode IDs (customer, merchant)
for col in id_features:
    X[col] = X[col].astype('category').cat.codes

# scale them
numerical_features += id_features

In [ ]:
X

In [ ]:
# step 6: preprocessing pipeline
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # keeps customer and merchant
)

In [ ]:
X

In [ ]:
# step 7: isolation forest pipeline
iso_forest_model = IsolationForest(
    n_estimators=100,       # number of trees
    contamination=y.mean(),    # expected fraud rate
    random_state=42         # reproducibility
)

In [ ]:
# step 8: full pipeline
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', iso_forest_model)
])

In [ ]:
# create training set
X_train = X[y == 0].copy()

In [ ]:
# step 9: train model
pipeline.fit(X_train)

In [ ]:
# step 10: get anomaly scores
scores = pipeline.decision_function(X)

# convert to predictions
y_pred = pipeline.predict(X)
y_pred = np.where(y_pred == -1, 1, 0) # convert: 1 = normal, -1 = anomaly -> 0/1

In [ ]:
# step 11: evaluate
print("ROC-AUC:", roc_auc_score(y, -scores))
print(classification_report(y, y_pred))

## Autoencoder

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class FraudAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
def anomaly_score(model, X_tensor):
    with torch.no_grad():
        recon = model(X_tensor)
        return ((recon - X_tensor) ** 2).mean(dim=1)

## Graph Neural Network

In [ ]:
from torch_geometric.nn import GCNConv

In [ ]:
# build graph
edge_index = torch.tensor([
    X['customer'].values,
    X['merchant'].values
], dtype=torch.long)

# node features


In [ ]:
class GNNModel(nn.Module):
    def __init__(self, in_feats, hidden, out_feats):
        super().__init__()
        self.conv1 = GCNConv(in_feats, hidden)
        self.conv2 = GCNConv(hidden, out_feats)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)